###Standard Connector - Autoloader for file based ingestion from cloud storage

#### Landing volume

We use a managed Unity Catalog volume as the landing location — this
is an S3 location where files get dropped into, without requiring an
external location / storage credential setup.


In [0]:
%sql
USE CATALOG dev;
USE SCHEMA dbx_course;

CREATE VOLUME IF NOT EXISTS dev.dbx_course.landing;


In [0]:
%python
# Folder structure inside the volume — one subfolder per source table.
# Auto Loader watches these paths for new files.
dbutils.fs.mkdirs("/Volumes/dev/dbx_course/landing/orders")
dbutils.fs.mkdirs("/Volumes/dev/dbx_course/landing/customers")


**Before running the cells below**: upload the two batch-1 files provided
with this lecture into the volume using Catalog Explorer
(*Catalog → dev → dbx_course → landing → orders / customers → Upload*):

- `batch1_orders.csv` → `/Volumes/dev/dbx_course/landing/orders/`
- `batch1_customers.csv` → `/Volumes/dev/dbx_course/landing/customers/`

This represents the 2024-01-20 drop


#### Auto Loader — orders

`cloudFiles.schemaLocation` is what enables schema inference *and* evolution —
Auto Loader writes its inferred schema there and checks new files against it.
`checkpointLocation` is what makes this incremental: on the next run, Auto
Loader only processes files it hasn't seen.

The `_metadata` column is how we implement Bronze convention
(`_ingest_timestamp`, `_source`, `_source_file`) from a real file source.


In [0]:
%python
from pyspark.sql.functions import col, lit

landing_path = "/Volumes/dev/dbx_course/landing"

orders_schema_location = f"{landing_path}/_schemas/orders"
orders_checkpoint_location = f"{landing_path}/_checkpoints/orders"

orders_raw = (
    spark.readStream
        .format("cloudFiles") #Structured Streaming source for Autoloader
        .option("cloudFiles.format", "csv") 
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", orders_schema_location)
        .option("header", "true")
        .load(f"{landing_path}/orders")
)

orders_bronze = (
    orders_raw
        .withColumn("_ingest_timestamp", col("_metadata.file_modification_time"))
        .withColumn("_source", lit("s3_landing_autoloader"))
        .withColumn("_source_file", col("_metadata.file_path"))
)

(orders_bronze.writeStream
    .option("checkpointLocation", orders_checkpoint_location)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("dev.dbx_course.bronze_orders_v2")
    .awaitTermination()
)


In [0]:
%sql
DESCRIBE TABLE dev.dbx_course.bronze_orders_v2;


In [0]:
%sql
SELECT order_id, customer_id, order_date, status, quantity, unit_price, discount_code,
       _ingest_timestamp, _source, _source_file, _rescued_data
FROM dev.dbx_course.bronze_orders_v2
ORDER BY order_id;


> Check out the schema location and checkpoint location


#### Auto Loader — customers

Same pattern as orders, applied to `bronze_customers_v2`. 


In [0]:
%python
customers_schema_location = f"{landing_path}/_schemas/customers"
customers_checkpoint_location = f"{landing_path}/_checkpoints/customers"

customers_raw = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", customers_schema_location)
        .option("header", "true")
        .load(f"{landing_path}/customers")
)

customers_bronze = (
    customers_raw
        .withColumn("_ingest_timestamp", col("_metadata.file_modification_time"))
        .withColumn("_source", lit("s3_landing_autoloader"))
        .withColumn("_source_file", col("_metadata.file_path"))
)

(customers_bronze.writeStream
    .option("checkpointLocation", customers_checkpoint_location)
    .option("mergeSchema", "true")
    .trigger(availableNow=True)
    .toTable("dev.dbx_course.bronze_customers_v2")
    .awaitTermination()
)


In [0]:
%sql
SELECT customer_id, customer_name, tier, region,
       _ingest_timestamp, _source, _source_file
FROM dev.dbx_course.bronze_customers_v2
ORDER BY customer_id;
